This notebook was created by Donna Faith Go.

In [1]:
# import sys
# !{sys.executable} -m pip install -qq -r requirements.txt

In [2]:
# standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import re

# webscraping
import requests

# data gathering
import yfinance as yf
import time
# import pandas_datareader.data as web
from datetime import datetime, timedelta

# ignore warnings
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

## Data Gathering

## Webscrape stock indices

In [3]:
# getting the stocks
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    )
}

response = requests.get(
    "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    headers=headers
)
response.raise_for_status()
tables = pd.read_html(response.text)

if len(tables) > 0:
    stocks_df = tables[0]

# randomly selecting 100 stocks
stocks_list = stocks_df['Symbol'].to_list()
stocks_df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [4]:
stocks_df.to_pickle("checkpoints/SP500 comapnies list.pkl")

## Saving to `.pkl` files

In [5]:
# pull individual stock data from yfinance
def download_info_per_stock(ticker, verbose=False, 
                            start_date='2000-01-01', 
                            end_date='2026-01-01'):
        try:
            # get data for the ticker
            ticker_data = yf.download(
                ticker,
                start=start_date,
                end=end_date,
                progress=False,
                timeout=120 # in case of slow internet, in seconds
            )
            return pd.DataFrame(ticker_data)
            
        except Exception as e:
            if verbose:
                print(f"Error downloading {ticker}.")
            return None

# saving individual stock data
def save_info_per_stock(ticker_list, delay=1, 
                        verbose=False, override=False,
                        start_date='2000-01-01', 
                        end_date='2026-01-01'):
    
    # create the data folder
    os.makedirs("data", exist_ok=True)

    for i in range(0, len(ticker_list)):
        # declare company name and filepath
        ticker_name = stocks_df['Security'][i]
        filepath = f"data/{ticker_name}.pkl"

        # skip if not override and file exists
        if override == False and os.path.exists(filepath):
            if verbose:
                print(f"Skipped {ticker_name}.")
                continue
        else:                    
            # get the data for each stock
            if verbose:
                print()
                print(f"Downloading for ticker: {ticker_list[i]}")
            ticker_data = download_info_per_stock(ticker_list[i],
                                                  start_date=start_date,
                                                  end_date=end_date)
    
            # saving data as a pkl file
            if ticker_data is not None and not ticker_data.empty:
                ticker_data.to_pickle(filepath)
                if verbose == True:
                    print(f"Saved data for {ticker_name}.")
            
            # avoid rate limiting
            time.sleep(delay)

    print("Done downloading all data!")

In [6]:
save_info_per_stock(stocks_list, end_date='2023-05-25')

Skipped 3M.
Skipped A. O. Smith.
Skipped Abbott Laboratories.
Skipped AbbVie.
Skipped Accenture.
Skipped Adobe Inc..
Skipped Advanced Micro Devices.
Skipped AES Corporation.
Skipped Aflac.
Skipped Agilent Technologies.
Skipped Air Products.
Skipped Airbnb.
Skipped Akamai Technologies.
Skipped Albemarle Corporation.
Skipped Alexandria Real Estate Equities.
Skipped Align Technology.
Skipped Allegion.
Skipped Alliant Energy.
Skipped Allstate.
Skipped Alphabet Inc. (Class A).
Skipped Alphabet Inc. (Class C).
Skipped Altria.
Skipped Amazon.
Skipped Amcor.
Skipped Ameren.
Skipped American Electric Power.
Skipped American Express.
Skipped American International Group.
Skipped American Tower.
Skipped American Water Works.
Skipped Ameriprise Financial.
Skipped Ametek.
Skipped Amgen.
Skipped Amphenol.
Skipped Analog Devices.
Skipped Aon plc.
Skipped APA Corporation.
Skipped Apollo Global Management.
Skipped Apple Inc..
Skipped Applied Materials.
Skipped AppLovin.
Skipped Aptiv.
Skipped Arch Capi


1 Failed download:
['BRK.B']: YFTzMissingError('possibly delisted; no timezone found')


Skipped Best Buy.
Skipped Bio-Techne.
Skipped Biogen.
Skipped BlackRock.
Skipped Blackstone Inc..
Skipped Block, Inc..
Skipped BNY Mellon.
Skipped Boeing.
Skipped Booking Holdings.
Skipped Boston Scientific.
Skipped Bristol Myers Squibb.
Skipped Broadcom.
Skipped Broadridge Financial Solutions.
Skipped Brown & Brown.




1 Failed download:
['BF.B']: YFPricesMissingError('possibly delisted; no price data found  (1d 2000-01-01 -> 2023-05-25)')


Skipped Builders FirstSource.
Skipped Bunge Global.
Skipped BXP, Inc..
Skipped C.H. Robinson.
Skipped Cadence Design Systems.
Skipped Camden Property Trust.
Skipped Campbell's Company (The).
Skipped Capital One.
Skipped Cardinal Health.
Skipped Carnival.
Skipped Carrier Global.
Skipped Carvana.
Skipped Caterpillar Inc..
Skipped Cboe Global Markets.
Skipped CBRE Group.
Skipped CDW Corporation.
Skipped Cencora.
Skipped Centene Corporation.
Skipped CenterPoint Energy.
Skipped CF Industries.
Skipped Charles River Laboratories.
Skipped Charles Schwab Corporation.
Skipped Charter Communications.
Skipped Chevron Corporation.
Skipped Chipotle Mexican Grill.
Skipped Chubb Limited.
Skipped Church & Dwight.
Skipped Ciena.
Skipped Cigna.
Skipped Cincinnati Financial.
Skipped Cintas.
Skipped Cisco.
Skipped Citigroup.
Skipped Citizens Financial Group.
Skipped Clorox.
Skipped CME Group.
Skipped CMS Energy.
Skipped Coca-Cola Company (The).
Skipped Cognizant.
Skipped Coinbase.
Skipped Colgate-Palmolive

...

Skipped SBA Communications.
Skipped Schlumberger.
Skipped Seagate Technology.
Skipped Sempra.
Skipped ServiceNow.
Skipped Sherwin-Williams.
Skipped Simon Property Group.
Skipped Skyworks Solutions.
Skipped J.M. Smucker Company (The).
Skipped Smurfit Westrock.
Skipped Snap-on.




1 Failed download:
['SOLV']: YFPricesMissingError('possibly delisted; no price data found  (1d 2000-01-01 -> 2023-05-25) (Yahoo error = "Data doesn\'t exist for startDate = 946702800, endDate = 1684987200")')


Skipped Southern Company.
Skipped Southwest Airlines.
Skipped Stanley Black & Decker.
Skipped Starbucks.
Skipped State Street Corporation.
Skipped Steel Dynamics.
Skipped Steris.
Skipped Stryker Corporation.
Skipped Supermicro.
Skipped Synchrony Financial.
Skipped Synopsys.
Skipped Sysco.
Skipped T-Mobile US.
Skipped T. Rowe Price.
Skipped Take-Two Interactive.
Skipped Tapestry, Inc..
Skipped Targa Resources.
Skipped Target Corporation.
Skipped TE Connectivity.
Skipped Teledyne Technologies.
Skipped Teradyne.
Skipped Tesla, Inc..
Skipped Texas Instruments.
Skipped Texas Pacific Land Corporation.
Skipped Textron.
Skipped Thermo Fisher Scientific.
Skipped TJX Companies.
Skipped TKO Group Holdings.
Skipped Trade Desk (The).
Skipped Tractor Supply.
Skipped Trane Technologies.
Skipped TransDigm Group.
Skipped Travelers Companies (The).
Skipped Trimble Inc..
Skipped Truist Financial.
Skipped Tyler Technologies.
Skipped Tyson Foods.
Skipped U.S. Bancorp.
Skipped Uber.
Skipped UDR, Inc..
Skipp


1 Failed download:
['VLTO']: YFPricesMissingError('possibly delisted; no price data found  (1d 2000-01-01 -> 2023-05-25) (Yahoo error = "Data doesn\'t exist for startDate = 946702800, endDate = 1684987200")')


Skipped Verisign.
Skipped Verisk Analytics.
Skipped Verizon.
Skipped Vertex Pharmaceuticals.
Skipped Viatris.
Skipped Vici Properties.
Skipped Visa Inc..
Skipped Vistra Corp..
Skipped Vulcan Materials Company.
Skipped W. R. Berkley Corporation.
Skipped W. W. Grainger.
Skipped Wabtec.
Skipped Walmart.
Skipped Walt Disney Company (The).
Skipped Warner Bros. Discovery.
Skipped Waste Management.
Skipped Waters Corporation.
Skipped WEC Energy Group.
Skipped Wells Fargo.
Skipped Welltower.
Skipped West Pharmaceutical Services.
Skipped Western Digital.
Skipped Weyerhaeuser.
Skipped Williams-Sonoma, Inc..
Skipped Williams Companies.
Skipped Willis Towers Watson.
Skipped Workday, Inc..
Skipped Wynn Resorts.
Skipped Xcel Energy.
Skipped Xylem Inc..
Skipped Yum! Brands.
Skipped Zebra Technologies.
Skipped Zimmer Biomet.
Skipped Zoetis.
Done downloading all data!


## Old Code

In [7]:
# getting closing prices for the 30 stocks with batching
start_date = '2000-01-01'
end_date = '2023-05-25'

def download_stocks_in_batches(tickers, batch_size=5, delay=1):
    """
    Download stock data in batches to avoid rate limiting
    """
    all_data = {}
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i + batch_size]
        print(f"Downloading batch {i//batch_size + 1}: {batch}")
        
        try:
            # Download the batch
            batch_data = yf.download(
                batch,
                start=start_date,
                end=end_date,
                progress=False
            )
            
            # Extract closing prices for this batch
            if not batch_data.empty and 'Close' in batch_data.columns:
                closes = batch_data['Close']
                if isinstance(closes, pd.Series):
                    all_data[batch[0]] = closes
                else:
                    for ticker in closes.columns:
                        all_data[ticker] = closes[ticker]
                print(f"Successfully downloaded {len(batch)} stocks")
            else:
                print(f"No data returned for batch: {batch}")
            
        except Exception as e:
            print(f"Error downloading batch {batch}: {e}")
        
        # Add delay to avoid rate limiting
        if i + batch_size < len(tickers):
            print(f"Waiting {delay} seconds before next batch...")
            time.sleep(delay)
    
    if all_data:
        return pd.DataFrame(all_data)
    else:
        return pd.DataFrame()

In [8]:
# # Download the volatility indices
# closing_df = download_stocks_in_batches(
#     ['^GSPC'], 
#     batch_size=5, 
#     delay=5
# )

# if not closing_df.empty:
#     closing_df.to_pickle('sp 500 data.pkl')